# CrewAI Hierarchical Process

In [1]:
"""
CrewAI Hierarchical Pattern - Contoh Sederhana
Menunjukkan cara kerja manager agent yang mengatur worker agents
"""

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field


In [2]:
# ============================================================================
# 1. SETUP LLM
# ============================================================================

llm = LLM(
    model="ollama/qwen3:0.6b-q4_K_M",
    base_url="http://localhost:11434",
    temperature=0.7
)

In [3]:
# ============================================================================
# 2. DEFINE TOOLS (Using BaseTool)
# ============================================================================

class SearchToolInput(BaseModel):
    """Input for SearchTool."""
    query: str = Field(..., description="Search query")

class SearchTool(BaseTool):
    name: str = "Search Tool"
    description: str = "Search for information on the internet"
    args_schema: Type[BaseModel] = SearchToolInput

    def _run(self, query: str) -> str:
        # Simulasi search
        return f"Hasil pencarian untuk '{query}': Ditemukan berbagai informasi tentang {query}."


class CalculatorToolInput(BaseModel):
    """Input for CalculatorTool."""
    expression: str = Field(..., description="Mathematical expression to calculate")

class CalculatorTool(BaseTool):
    name: str = "Calculator Tool"
    description: str = "Calculate mathematical expressions"
    args_schema: Type[BaseModel] = CalculatorToolInput

    def _run(self, expression: str) -> str:
        try:
            result = eval(expression)
            return f"Hasil kalkulasi: {result}"
        except Exception as e:
            return f"Error dalam kalkulasi: {str(e)}"


class WriteFileToolInput(BaseModel):
    """Input for WriteFileTool."""
    filename: str = Field(..., description="Name of the file to write")
    content: str = Field(..., description="Content to write to the file")

class WriteFileTool(BaseTool):
    name: str = "Write File Tool"
    description: str = "Write content to a file"
    args_schema: Type[BaseModel] = WriteFileToolInput

    def _run(self, filename: str, content: str) -> str:
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(content)
            return f"File '{filename}' berhasil ditulis"
        except Exception as e:
            return f"Error menulis file: {str(e)}"


# Initialize tools
search_tool = SearchTool()
calculator_tool = CalculatorTool()
write_file_tool = WriteFileTool()


In [4]:
# ============================================================================
# 3. CREATE WORKER AGENTS
# ============================================================================

# Researcher Agent - bertugas mencari informasi
researcher = Agent(
    role="Researcher",
    goal="Mencari dan mengumpulkan informasi yang relevan tentang topik yang diminta",
    backstory="""Anda adalah seorang peneliti berpengalaman yang ahli dalam 
    mencari informasi akurat dan relevan. Anda selalu memberikan data yang 
    terverifikasi dan dapat dipercaya.""",
    tools=[search_tool],
    llm=llm,
    max_iter=3,  # Maksimal 3 iterasi untuk researcher
    verbose=True,
    allow_delegation=False  # Worker tidak bisa delegate
)

# Analyst Agent - bertugas menganalisis data
analyst = Agent(
    role="Data Analyst",
    goal="Menganalisis informasi yang dikumpulkan dan memberikan insight yang berguna",
    backstory="""Anda adalah seorang analis data yang handal. Anda mampu 
    mengidentifikasi pola, tren, dan memberikan kesimpulan yang actionable 
    dari data yang ada.""",
    tools=[calculator_tool],
    llm=llm,
    max_iter=3,  # Maksimal 3 iterasi untuk analyst
    verbose=True,
    allow_delegation=False  # Worker tidak bisa delegate
)

# Writer Agent - bertugas menulis laporan
writer = Agent(
    role="Technical Writer",
    goal="Menulis laporan yang jelas, terstruktur, dan mudah dipahami",
    backstory="""Anda adalah seorang technical writer profesional yang 
    mampu mengubah informasi kompleks menjadi dokumen yang mudah dipahami 
    oleh berbagai audience.""",
    tools=[write_file_tool],
    llm=llm,
    max_iter=2,  # Maksimal 2 iterasi untuk writer
    verbose=True,
    allow_delegation=False  # Worker tidak bisa delegate
)


In [5]:
# ============================================================================
# 4. CREATE MANAGER AGENT (OPTIONAL - Auto-created if not specified)
# ============================================================================

# Manager agent akan otomatis dibuat oleh CrewAI jika tidak didefinisikan
# Tapi kita bisa membuat custom manager juga:

manager = Agent(
    role="Project Manager",
    goal="Mengkoordinasikan tim untuk menyelesaikan proyek dengan efisien",
    backstory="""Anda adalah seorang project manager berpengalaman yang 
    ahli dalam mendelegasikan tugas, mengatur prioritas, dan memastikan 
    semua anggota tim bekerja secara optimal untuk mencapai tujuan bersama.""",
    llm=llm,
    max_iter=5,  # Manager bisa punya lebih banyak iterasi
    verbose=True,
    allow_delegation=True  # Manager bisa delegate ke workers
)

In [6]:
# ============================================================================
# 5. CREATE TASKS
# ============================================================================

# Task utama yang akan di-handle oleh manager
main_task = Task(
    description="""
    Lakukan riset mendalam tentang "AI Agents dan LangGraph" kemudian:
    1. Kumpulkan informasi tentang konsep dasar, use cases, dan implementasi
    2. Analisis kelebihan dan kekurangan dari teknologi ini
    3. Buat laporan komprehensif dalam format markdown
    
    Pastikan laporan mencakup:
    - Penjelasan konsep
    - Contoh penggunaan
    - Analisis pro/cons
    - Rekomendasi implementasi
    """,
    agent=manager,  # Task di-assign ke manager
    expected_output="Laporan lengkap dalam format markdown tentang AI Agents dan LangGraph"
)

In [7]:
# ============================================================================
# 6. CREATE HIERARCHICAL CREW
# ============================================================================

# Cara 1: Dengan manager yang didefinisikan sendiri
crew_with_custom_manager = Crew(
    agents=[researcher, analyst, writer],  # Manager TIDAK dimasukkan ke agents list
    tasks=[main_task],
    process=Process.hierarchical,  # Mode hierarchical
    verbose=True,
    manager_agent=manager,  # Gunakan custom manager
    memory=False  # Disable memory untuk simplicity
)

# Cara 2: Tanpa custom manager (auto-created)
crew_with_auto_manager = Crew(
    agents=[researcher, analyst, writer],  # Hanya workers
    tasks=[main_task],
    process=Process.hierarchical,  # Mode hierarchical
    verbose=True,
    tracing=True,
    manager_llm=llm,  # LLM untuk auto-generated manager
    memory=False
)


In [8]:
crew_with_custom_manager.kickoff()  # Mulai proses CrewAI

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 01d273a4-209e-48d1-a8af-32f82e0a2781                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Lakukan riset mendalam tentang "AI Agents dan LangGraph" kemudian:                                         │
│      1. Kumpulkan informasi tentang konsep dasar, use cases, dan implementasi                                   │
│      2. Analisis kelebihan dan kekurangan dari teknologi ini                                                    │
│      3. Buat laporan komprehensif dalam format markdown                                                         │
│                                                                                                                 │
│      Pastikan laporan mencakup:                                                                                 │
│      - Penjelasan konsep                                                                                        │
│      - Contoh penggunaan                                                                                        │
│      - Analisis pro/cons                                                                                        │
│      - Rekomendasi implementasi                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```                                                                                                            │
│  # Laporan Komprehensif tentang AI Agents dan LangGraph                                                         │
│                                                                                                                 │
│  ## Penjelasan Konsep                                                                                           │
│  AI Agents adalah program inti yang mengontrol dan mengelola sistem AI, berbasis pada algoritma dan interaksi   │
│  antaranggota AI. LangGraph adalah metode sistematis untuk memahami, mengelola, dan memprediksi perubahan data  │
│  melalui alur logika dan alur interaksi antar informasi.                                                        │
│                                                                                                                 │
│  ## Contoh Penggunaan                                                                                           │
│  - **LangGraph** dapat digunakan untuk analisis data multivariabel melalui alur interaksi antar informasi,      │
│  seperti dalam perusahaan AI.                                                                                   │
│  - **AI Agents** dapat diimplementasikan untuk memproses data multivariabel secara otomatis melalui algoritma   │
│  seperti Kalman Filters.                                                                                        │
│                                                                                                                 │
│  ## Analisis Pro/Cons                                                                                           │
│  - **Kelebihan**:                                                                                               │
│    - Efisien dalam pengelolaan data multivariabel.                                                              │
│    - Kemampuan untuk memprediksi perubahan data secara real-time.                                               │
│                                                                                                                 │
│  - **Kekurangan**:                                                                                              │
│    - Seringkali terbatas pada informasi yang diterima secara langsung.                                          │
│    - Kompleksitas dalam implementasi terhadap algoritma.                                                        │
│                                                                                                                 │
│  ## Rekomendasi Implementasi                                                                                    │
│  1. **LangGraph**: Implementasikan untuk memahami interaksi antar informasi.                                    │
│  2. **AI Agents**: Pilih algoritma sesuai kebutuhan data multivariabel.                                         │
│  3. **Implementasi**: Gunakan langkah-langkah alur interaksi antar informasi untuk memastikan efisiensi.        │
│  ```                                                                                                            │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

/home/ilham/Documents/python/crewai-vs-langgraph/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 0672ec00-8e5d-4b41-be77-b3ce24ee78b1                                                                     │
│  Agent: Project Manager                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw='```  \n# Laporan Komprehensif tentang AI Agents dan LangGraph  \n\n## Penjelasan Konsep  \nAI Agents adalah program inti yang mengontrol dan mengelola sistem AI, berbasis pada algoritma dan interaksi antaranggota AI. LangGraph adalah metode sistematis untuk memahami, mengelola, dan memprediksi perubahan data melalui alur logika dan alur interaksi antar informasi.  \n\n## Contoh Penggunaan  \n- **LangGraph** dapat digunakan untuk analisis data multivariabel melalui alur interaksi antar informasi, seperti dalam perusahaan AI.  \n- **AI Agents** dapat diimplementasikan untuk memproses data multivariabel secara otomatis melalui algoritma seperti Kalman Filters.  \n\n## Analisis Pro/Cons  \n- **Kelebihan**:  \n  - Efisien dalam pengelolaan data multivariabel.  \n  - Kemampuan untuk memprediksi perubahan data secara real-time.  \n\n- **Kekurangan**:  \n  - Seringkali terbatas pada informasi yang diterima secara langsung.  \n  - Kompleksitas dalam implementasi terhadap algorit

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 01d273a4-209e-48d1-a8af-32f82e0a2781                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```                                                                                              │
│  # Laporan Komprehensif tentang AI Agents dan LangGraph                                                         │
│                                                                                                                 │
│  ## Penjelasan Konsep                                                                                           │
│  AI Agents adalah program inti yang mengontrol dan mengelola sistem AI, berbasis pada algoritma dan interaksi   │
│  antaranggota AI. LangGraph adalah metode sistematis untuk memahami, mengelola, dan memprediksi perubahan data  │
│  melalui alur logika dan alur interaksi antar informasi.                                                        │
│                                                                                                                 │
│  ## Contoh Penggunaan                                                                                           │
│  - **LangGraph** dapat digunakan untuk analisis data multivariabel melalui alur interaksi antar informasi,      │
│  seperti dalam perusahaan AI.                                                                                   │
│  - **AI Agents** dapat diimplementasikan untuk memproses data multivariabel secara otomatis melalui algoritma   │
│  seperti Kalman Filters.                                                                                        │
│                                                                                                                 │
│  ## Analisis Pro/Cons                                                                                           │
│  - **Kelebihan**:                                                                                               │
│    - Efisien dalam pengelolaan data multivariabel.                                                              │
│    - Kemampuan untuk memprediksi perubahan data secara real-time.                                               │
│                                                                                                                 │
│  - **Kekurangan**:                                                                                              │
│    - Seringkali terbatas pada informasi yang diterima secara langsung.                                          │
│    - Kompleksitas dalam implementasi terhadap algoritma.                                                        │
│                                                                                                                 │
│  ## Rekomendasi Implementasi                                                                                    │
│  1. **LangGraph**: Implementasikan untuk memahami interaksi antar informasi.                                    │
│  2. **AI Agents**: Pilih algoritma sesuai kebutuhan data multivariabel.                                         │
│  3. **Implementasi**: Gunakan langkah-langkah alur interaksi antar informasi untuk memastikan efisiensi.        │
│  ```                                                                                                            │
│                                                       

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 90042ac7-0b86-4911-8b24-1bd1ee0d08d4                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/90042ac7-0b86-491 │
│ 1-8b24-1bd1ee0d08d4?access_code=TRACE-10046709d4                             │
│ 🔑 Access Code: TRACE-10046709d4                                             │
╰──────────────────────────────────────────────────────────────────────────────╯
